🏀 NBA Performance & Compensation: A Predictive Analysis
Author: Walter Hampton III

Objective: To determine if on-court performance metrics can accurately predict whether an NBA player’s salary exceeds the league average ($7.7M). This project bridges the gap between raw box-score statistics and financial valuation.

The Challenge: Assessing "value" in the NBA is complex. Beyond pure scoring, player compensation is influenced by rookie-scale contracts, defensive utility, and market-driven usage rates.

Methodology: * Data Source: Comprehensive 2020-2021 NBA season statistics and salary data. The data was pulled 51 games into the 2020-2021 NBA season.

Approach: Feature engineering and Machine Learning classification to identify "value-driving" indicators.

Goal: To build a model that isolates performance-based value from external contractual noise.

In [18]:
import sys
import os
os.chdir('C:\\Users\\wally')
import numpy as np
from scipy.io import arff
import pandas as pd
%matplotlib widget
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import metrics
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
import matplotlib.patches as mpatches
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, 
    confusion_matrix, 
    roc_auc_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    classification_report, 
    roc_curve, 
    auc, 
    precision_recall_curve, 
    average_precision_score, 
    log_loss,
    # The new display classes that replaced the 'plot_' functions:
    ConfusionMatrixDisplay, 
    RocCurveDisplay, 
)
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import cross_val_score
plt.rcParams.update(plt.rcParamsDefault)
%matplotlib inline
os.chdir('C:\\Users\\wally\\OneDrive\\Documents')

## 🛠️ Data Engineering & Preprocessing
In this stage, I focus on **Data Integrity**.
1. **Feature Engineering:** Players with hybrid roles (e.g., "PF-C") are normalized to their primary position to ensure clean categorical analysis.
2. **Filtering Noise:** Financial "leakage" variables (like guaranteed money or future projections) are removed to ensure the model relies purely on performance stats. (Filtered later on in the notebook)
3. **Handling Scarcity:** Addressed missing values and normalized statistics to a "Per Game" basis where necessary.

In [41]:
ss = pd.read_csv('C:\\Users\\wally\\OneDrive\\Documents\\Data Analytics Projects\\NBA Stats and Salaries.csv')
ss = ss.rename(columns = {'salary_above_average': 'salary'})
ss["PPG"] = ss["PTS"]/ss["G"]
# 1. Clean the Position column
# This handles 'C-PF' -> 'C', 'PF-C' -> 'PF', and 'PG' -> 'PG'
ss['Pos_Cleaned'] = ss['Pos'].str.split('-').str[0]
ss.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439 entries, 0 to 438
Data columns (total 38 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Rk                439 non-null    int64  
 1   player_name       439 non-null    object 
 2   Pos               439 non-null    object 
 3   Age               439 non-null    int64  
 4   Tm                439 non-null    object 
 5   G                 439 non-null    int64  
 6   GS                439 non-null    int64  
 7   MP                439 non-null    int64  
 8   FG                439 non-null    int64  
 9   FGA               439 non-null    int64  
 10  FG%               439 non-null    float64
 11  3P                439 non-null    int64  
 12  3PA               439 non-null    int64  
 13  3P%               439 non-null    float64
 14  2P                439 non-null    int64  
 15  2PA               439 non-null    int64  
 16  2P%               439 non-null    float64
 1

Variable Description:
1. RK = Alphabetical Rank Among Players
2. player_name = Player Name
3. Pos = Player Position
4. Age = Age
5. Tm = Team
6. G = Games Played
7. GS = Games Started
8. MP = Minutes Played
9. FG = Field Goals Made
10. FGA = Field Goals Attempted
11. FG% = Field Goal Percentage
12. 3P = 3 Pointers Made
13. 3PA = 3 Pointers Attempted
14. 3P% = 3 Point Percentage
15. 2P = 2 Pointers Made
16. 2PA = 2 Pointers Attempted
17. 2P% = 2 Point Percentage
18. eFG% = Effective Field Goal Percentage (Adjusts for 3 points being 1 more point than two pointers)
19. FT = Free Throws Made
20. FTA = Free Throws Attempted
21. FT% = Free Throw Percentage
22. ORB = Offensive Rebounds
23. DRB = Defensive Rebounds
24. TRB = Total Rebounds
25. AST = Assists
26. STL = Steals
27. BLK = Blocks
28. TOV = Turnovers
29. PF = Personal Fouls
30. PTS = Points
31. salary_rank = Rank based on salary for 2020-21 season
32. salary_20-21 = Salary for 2020-21 season
33. salary_21-22 = Salary for 2021-22 season
34. signed_using = Type of contract
35. guaranteed_money = Guaranteed money for length of contract
36. salary = Indicates whether the player earns more than $7689656 USD for the 2020-21 season. 1 = Above Average, 0 = Below Average
37. PPG = Points per Game
38. Pos_cleaned = Positions containing two positions condensed into one, prefix of two positions was taken

## 🔍 Exploratory Data Analysis (EDA)
Before modeling, I explore the underlying distributions to identify trends and outliers. 
* **Points Per Game vs. Salary:** Using the PPG metric to find an overall trend.
* **Distribution of Age:** Understanding the league's lifecycle.
* **Salary by Position:** Testing the hypothesis: *"Are playmakers (Guards) paid more than rim protectors (Centers) in the modern NBA?"*


In [42]:
import plotly.express as px

# 1. Remove text='player_name' to clear the static labels
fig = px.scatter(ss, x='PPG', y='salary_rank', 
                 hover_name='player_name', # This ensures the name appears ONLY on hover
                 title='NBA Salary Rank vs PPG',
                 color='salary_rank',
                 template='plotly_white',
                 labels={
                     'salary_rank': 'Salary Rank'} 
                     )

# 2. (Optional) Customize the tooltip to show more stats
# This adds PPG and salary_rank to the hover box with cleaner labels
fig.update_traces(
    hovertemplate="<b>%{hovertext}</b><br>PPG: %{x}<br>Salary Rank: %{y}<extra></extra>"
)
fig.update_layout(
    yaxis_title="Salary Rank",
    xaxis_title="Average Points Per Game",
    font=dict(size=14) # Makes it easier for recruiters to read
)

fig.update_layout(
    legend_title_text='Salary Rank'
)

fig.show()

In [43]:
fig_age = px.histogram(ss, x='Age', 
                       title='NBA Player Age Distribution',
                       labels={'Age': 'Player Age', 'y' : 'Number of Players'},
                       template='plotly_white',
                       color_discrete_sequence=['indianred'])

fig_age.update_layout(yaxis_title="Number of Players", bargap=0.1)
fig_age.show()

In [52]:
# 2. Calculate the average salary rank (or actual salary)
# This uses the cleaned column for the group
avg_salary = ss.groupby('Pos_Cleaned')['salary_20-21'].mean().reset_index()

# 3. Sort for better visualization (Optional but recommended)
avg_salary = avg_salary.sort_values('salary_20-21', ascending=False)

print(avg_salary)
# 1. Calculate the mean salary rank per position
avg_salary = ss.groupby('Pos_Cleaned')['salary_20-21'].mean().reset_index()

# 2. Sort the values so the highest paid is first (Senior Analyst move!)
avg_salary = avg_salary.sort_values('salary_20-21', ascending=False)

# 3. Create the Plot
fig = px.bar(
    avg_salary, 
    x='Pos_Cleaned', 
    y='salary_20-21',
    color='salary_20-21',
    color_continuous_scale='Viridis',
    title='Average Salary by Position (2020-21 Season)',
    labels={'salary_20-21': 'Avg Salary', 'Pos_Cleaned': 'Position'},
    template='plotly_white'
)

# 4. Add a "League Average" reference line
league_avg = ss['salary_20-21'].mean()
fig.add_hline(y=league_avg, line_dash="dash", line_color="red", 
              annotation_text=f"League Average: {league_avg:.0f}", 
              annotation_position="bottom right")

fig.show()

  Pos_Cleaned  salary_20-21
2          PG  1.168520e+07
1          PF  8.268328e+06
0           C  8.110683e+06
3          SF  7.758445e+06
4          SG  6.729619e+06


## 🩺 Diagnostic Analysis: Feature Selection
To build a robust model, I perform a diagnostic check on feature relationships.
* **Correlation Magnitude:** Identifying the top 10 stats most strongly tied to salary (magnitude check).
* **Games Played vs. Salary Rank:** Identifying players who outperformed vs underperformed against expectations.
* **Multicollinearity:** Checking for redundancy between stats (e.g., Points vs. Field Goals) to ensure the model remains efficient and interpretable.

In [45]:
# 1. Filter numeric stats only and exclude financial 'leaks'
exclude = ['salary_21-22', 'guaranteed_money', 'salary', 'Rk', 'salary_rank', 'PTS', 'FG', 'FGA', 'FTA', '2PA', '3PA']
numeric_df = ss.select_dtypes(include=[np.number]).drop(columns=exclude)

# 2. Find top 10 stats correlated with salary
# We use absolute value to find the strongest predictors
correlations = numeric_df.corr()['salary_20-21'].abs().sort_values(ascending=False)
top_10_stats = correlations.drop('salary_20-21').head(10).index.tolist()

# 3. CRITICAL: Put Salary FIRST in the list to place it at the top-left
ordered_cols = ['salary_20-21'] + top_10_stats
final_corr_matrix = numeric_df[ordered_cols].corr()

# 4. Plotly Heatmap
fig_heat = px.imshow(
    final_corr_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    title="Top 10 Player Stats Correlated with Salary (2020-21)",
    labels={
        'color': 'Correlation',
        'x': 'Player Stats',
        'y': 'Player Stats'
    },
    template='plotly_white'
)

fig_heat.show()

In [46]:
fig_avail = px.scatter(
    ss, x='G', y='salary_rank', 
    hover_name='player_name',
    size='MP', # Size dots by Minutes Played
    color='salary_rank',
    title='Diagnostic: Games Played (G) vs. Salary Rank',
    labels={'G': 'Games Played', 'salary_rank': 'Salary Rank'},
    template='plotly_white'
)

# Add a vertical line at 41 games (half-season) to show the threshold
fig_avail.add_vline(x=25, line_dash="dash", line_color="red", 
                   annotation_text="Half Season Threshold (Dot Size = MP)")

fig_avail.show()

In [47]:
# Choose the top 3 predictors found in the heatmap (e.g., PPG, MP, TOP)
top_predictors = ['PPG', 'AST', 'TOV'] 

fig_pair = px.scatter_matrix(
    ss,
    dimensions=top_predictors,
    color='salary_rank',
    hover_name='player_name',
    title="Multicollinearity Diagnostic: Relationship Between Top Predictors",
    labels={'salary_rank': 'Salary Rank'},
    template='plotly_white'
)

# Reduce marker size for clarity
fig_pair.update_traces(diagonal_visible=False, marker=dict(size=4))
fig_pair.show()

## 🤖 Predictive Modeling
I compare three distinct mathematical approaches to find the most accurate predictor:
1. **Random Forest:** Captures non-linear relationships (e.g., the "Superstar Effect" where double the points might triple the salary).
2. **Gradient Boosting:** An iterative optimizer that refines predictions based on previous errors.
3. **Ridge Regression:** A linear approach with L2 regularization to handle the high correlation between basketball stats.

In [54]:
# 2. Data Engineering: Robust Position Cleaning
# Handles 'C-PF' -> 'C' and 'PF-C' -> 'PF' correctly
ss['Pos_Cleaned'] = ss['Pos'].str.split('-').str[0]

# 3. Feature Selection & Target Definition
# We exclude financial 'leaks' and non-numeric identifiers
target = 'salary_20-21'
exclude = ['salary_21-22', 'guaranteed_money', 'salary_above_average', 'Rk', 'salary_rank', 'player_name', 'Tm', 'Pos', 'signed_using']

# Drop missing values in target and select numeric stats
df = ss.dropna(subset=[target])
numeric_df = df.select_dtypes(include=[np.number])
X = numeric_df.drop(columns=[col for col in exclude if col in numeric_df.columns] + [target])
y = numeric_df[target]

# 4. Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Define the 3 Models
# Using a Pipeline for Ridge ensures we scale the data properly without leakage
models = {
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    "Ridge Regression": Pipeline([
        ('scaler', StandardScaler()), 
        ('model', Ridge(alpha=1.0, solver='cholesky')) # 'cholesky' avoids the sym_pos error
    ])
}

# 6. Evaluation Loop
results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    
    # Calculate Metrics
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    results.append({"Model": name, "R2": r2, "MAE": mae})

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)

# --- RESULTS PLOT 1: Model Benchmarking ---
fig_bench = px.bar(
    results_df, x='Model', y='R2', color='Model',
    title="Model Comparison: Predicting NBA Salaries (R² Score)",
    labels={'R2': 'Accuracy (R²)'},
    template='plotly_white'
)
fig_bench.show()

# --- RESULTS PLOT 2: Best Model Prediction Accuracy ---
best_model_name = results_df.iloc[0]['Model']
best_preds = predictions[best_model_name]
test_players = df.loc[X_test.index, 'player_name']

fig_res = px.scatter(
    x=y_test, y=best_preds, hover_name=test_players,
    title=f"Results: Actual vs. Predicted Salary ({best_model_name})",
    labels={'x': 'Actual Salary ($)', 'y': 'Model Prediction ($)'},
    template='plotly_white', opacity=0.7
)
# Add the 'Perfect Match' identity line
fig_res.add_trace(go.Scatter(x=[y_test.min(), y_test.max()], y=[y_test.min(), y_test.max()],
                             mode='lines', line=dict(color='red', dash='dash'), name='Perfect Match'))
fig_res.show()
display(results_df)

# 1. Create a dictionary to hold importance values
importance_data = {'Feature': X.columns}

# 2. Extract Feature Importance from Random Forest
importance_data['Random Forest'] = models["Random Forest"].feature_importances_

# 3. Extract Feature Importance from Gradient Boosting
importance_data['Gradient Boosting'] = models["Gradient Boosting"].feature_importances_

# 4. Extract Coefficients from Ridge Regression 
# Note: We use .abs() because magnitude indicates importance. 
# We access 'model' because it is inside a Pipeline.
ridge_coefs = np.abs(models["Ridge Regression"].named_steps['model'].coef_)
importance_data['Ridge (Abs Coef)'] = ridge_coefs

# 5. Create the DataFrame
importance_table = pd.DataFrame(importance_data)

# 6. Sort by the best performing model (usually Random Forest) to see the top drivers
importance_table = importance_table.sort_values(by='Random Forest', ascending=False).reset_index(drop=True)

# 7. Display the Top 10 Features
print("\n--- Top 10 Most Important Features Across Models ---")
display(importance_table.head(10).style.background_gradient(cmap='Blues'))

,Model,R2,MAE
0,Random Forest,0.695979,2.767165e+06
1,Gradient Boosting,0.654369,2.937780e+06
2,Ridge Regression,0.544220,3.443908e+06



--- Top 10 Most Important Features Across Models ---


,Feature,Random Forest,Gradient Boosting,Ridge (Abs Coef)
0,salary,0.624167,0.606143,4532870.614687
1,PPG,0.168003,0.178372,4992846.507141
2,Age,0.051502,0.075830,1310577.863733
3,AST,0.019245,0.020113,2420992.750876
4,FT%,0.012033,0.009591,172190.536581
5,ORB,0.010555,0.009267,290115.967300
6,TOV,0.010494,0.015230,346831.230429
7,2P%,0.009323,0.005985,139458.959034
8,G,0.008196,0.019959,534954.339297
9,BLK,0.006857,0.003259,865721.388328


## 🏆 Final Results & Model Evaluation
The models are evaluated on an 80/20 train/test split.
* **Accuracy:** Measured via $R^2$ and Mean Absolute Error (MAE).
* **Feature Importance:** Visualizing which on-court actions (Scoring, Playmaking, or Availability) actually dictate a player's paycheck.

### Conclusion & Next Steps
The analysis confirms that while **Points (PTS)** are a primary driver, **Age** and **Minutes Played** act as significant filters for salary. 
**Future Work:** - Incorporate "Advanced Stats" (True Shooting %, Player Efficiency Rating).
- Analyze the "Contract Year" effect (do players perform better when their contract is expiring?).